# Grouping and Aggregating Data with SQL

## Introduction: What is Grouping and Aggregation?

Up to this point in Unit 3, every query you have written operates row-by-row. You learned how to retrieve specific columns using `SELECT`, filter rows using `WHERE`, transform individual values using operators and functions, and sort output using `ORDER BY`. However, corporate managers, financial auditors, and executive leaders rarely review raw, transaction data. Instead, they require summarized financial statements such as total monthly sales, average transaction size per customer, or inventory counts grouped by warehouse location.

In data analytics, **grouping and aggregation** is the process of dividing a dataset into distinct category "piles" and calculating summary statistics for each pile.

### The Dollar-Counting Analogy

To understand the intuition behind grouping and aggregating, imagine a simple physical task: counting cash. Suppose you are handed a stack of \$1, \$5, \$10, and \$20 bills and asked to compute the total dollar value for each denomination. How would you proceed?

You would probably perform a two-step process:
1. **Grouping (Sorting into Piles):** You separate the bills into distinct piles based on their denomination (\$1 bills in one pile, \$5 bills in another, and so on).
2. **Aggregation (Calculating Summary Statistics):** Once the bills are sorted into piles, you count the number of bills or sum the total value within each pile.

```
+-----------------------------------------------------------------------+
|                              RAW DATA                                 |
|         Mixed Bills: [$1, $5, $10, $1, $20, $5, $1, $10]              |
+-----------------------------------------------------------------------+
                                   |
                                   v  GROUP BY Denomination
+------------------+------------------+------------------+--------------+
|   $1 Bill Pile   |   $5 Bill Pile   |  $10 Bill Pile   | $20 Bill Pile|
|   [$1, $1, $1]   |     [$5, $5]     |    [$10, $10]    |    [$20]     |
+------------------+------------------+------------------+--------------+
                                   |
                                   v  AGGREGATE (COUNT / SUM)
+------------------+------------------+------------------+---------------+
| Count: 3 ($3)    | Count: 2 ($10)   | Count: 2 ($20)   | Count: 1 ($20)|
+------------------+------------------+------------------+---------------+
```

This two-step process is the exact conceptual framework used in relational database querying. In SQL, the **`GROUP BY`** clause divides table rows into separate "piles". Once rows are grouped, SQL applies **aggregate functions** (`SUM`, `COUNT`, `AVG`, `MIN`, `MAX`) to calculate a single summary result for each group.

### The Excel Pivot Table Parallel

If you have used Microsoft Excel, you have already performed grouping and aggregation. An Excel **pivot table** performs the exact same operation as SQL's `GROUP BY` clause:
* Placing a field in the **rows** area of a Pivot Table is identical to specifying that field in a SQL `GROUP BY` clause.
* Placing a field in the **values** area (e.g., `Sum of Sales` or `Count of Orders`) is identical to applying a SQL aggregate function.

> **The Power of Relational Aggregation**  
> While Excel pivot tables allow quick drag-and-drop summaries, they require loading raw data into spreadsheet memory. In enterprise databases containing millions of credit card charges or ledger transactions, SQL's `GROUP BY` clause executes aggregation directly on the database engine server in seconds, returning compact summary tables without spreadsheet memory constraints.

---

## Identifying Categories: SELECT DISTINCT vs. GROUP BY

Before you can summarize data across categories, you must be able to identify the categories present in a dataset.

### Listing Unique Values with `SELECT DISTINCT`

Suppose you want to know all countries to which *Northwind Traders* has shipped customer orders. If you query the `ShipCountry` column using a standard `SELECT` statement, SQL returns 830 rows—one for every historical order—containing hundreds of duplicate country names.

To collapse duplicate values and display each unique category name once, use the **`DISTINCT`** keyword immediately after `SELECT`:

```sql
SELECT DISTINCT ShipCountry
FROM Orders
```

This query scans the `Orders` table and returns a clean list of 21 unique shipping countries (e.g., `Argentina`, `Austria`, `Belgium`, `Brazil`).

### Multi-Column Unique Combinations

You can pass multiple columns to `SELECT DISTINCT`. When you list multiple fields, SQL identifies every unique *combination* of values across those columns. 

For example, to determine all unique year-and-month calendar periods in which orders were placed:

```sql
SELECT DISTINCT 
    YEAR(OrderDate) AS [Year], 
    MONTH(OrderDate) AS [Month]
FROM Orders
ORDER BY [Year], [Month]
```

*Output (First 6 rows):*
| Year | Month |
| :--- | :--- |
| 1996 | 7 |
| 1996 | 8 |
| 1996 | 9 |
| 1996 | 10 |
| 1996 | 11 |
| 1996 | 12 |

SQL creates a distinct category key for every unique combination of year and month. This multi-column category listing forms the foundation for multi-period financial reporting.

### `SELECT DISTINCT` vs. `GROUP BY`

You can achieve the same unique country list using a `GROUP BY` clause without an aggregate function:

```sql
SELECT ShipCountry
FROM Orders
GROUP BY ShipCountry
```

While both queries return the same 21 unique country names, their analytical purpose differs:
* Use **`SELECT DISTINCT`** when your sole goal is to list unique category names or remove duplicates.
* Use **`GROUP BY`** when you intend to calculate summary statistics (`COUNT`, `SUM`, `AVG`) for each unique category.

---

## Summarizing Data by Category: The GROUP BY Clause

To compute summary statistics for each category in a table, combine the `GROUP BY` clause with aggregate functions in the `SELECT` statement.

### Basic `GROUP BY` Syntax

The standard syntax for a query with `GROUP BY` is:

```sql
SELECT GroupColumn, AGGREGATE_FUNCTION(ValueColumn) AS Alias
FROM TableName
GROUP BY GroupColumn
```

When SQL executes a `GROUP BY` query, it processes the statement in three distinct phases:
1. **`FROM` Phase:** Identifies the database source table.
2. **`GROUP BY` Phase:** Scans the target column, identifies all unique category values, and sorts the individual table rows into separate category "piles".
3. **`SELECT` Phase:** For each category pile, evaluates the aggregate function and outputs one summary row containing the category name and its calculated summary metric.

### Counting Records by Category

#### Example 1: Orders Placed per Customer

How many total orders has each customer placed with Northwind Traders? 

To answer this question, we group the `Orders` table by `CustomerID` and count the rows in each customer's pile using `COUNT(*)`:

```sql
SELECT 
    CustomerID, 
    COUNT(*) AS NumOrders
FROM Orders
GROUP BY CustomerID
```

*Output (Excerpt):*
| CustomerID | NumOrders |
| :--- | :--- |
| ALFKI | 6 |
| ANATR | 4 |
| ANTON | 5 |
| AOUTR | 2 |
| BERGS | 18 |

#### Example 2: Employee Staffing Headcount by Job Title

HR executives frequently monitor workforce distribution across job roles. How many employees hold each official job title in the `Employees` table?

```sql
SELECT 
    Title, 
    COUNT(*) AS EmployeeCount
FROM Employees
GROUP BY Title
```

*Output:*
| Title | EmployeeCount |
| :--- | :---: |
| Inside Sales Coordinator | 1 |
| Sales Representative | 6 |
| Sales Manager | 1 |
| Vice President, Sales | 1 |

### Summing Quantities by Category

#### Example 3: Total Unit Volume Sold per Product

Marketing managers tracking sales volume need to know the total physical quantity sold for every product item. The `OrderDetails` table contains one row per order per item ordered. Each row records a specific line item on an order (`ProductID`, `UnitPrice`, `Quantity`, `Discount`). 

To calculate total units sold per product, group by `ProductID` and apply the `SUM()` aggregate function to the `Quantity` column:

```sql
SELECT 
    ProductID, 
    SUM(Quantity) AS TotalQtySold
FROM OrderDetails
GROUP BY ProductID
ORDER BY ProductID;
```

*Output (Excerpt):*
| ProductID | TotalQtySold |
| :--- | :---: |
| 1 | 828 |
| 2 | 1057 |
| 3 | 328 |
| 4 | 453 |
| 5 | 298 |

### Improving Reports by Merging Parent Tables

In the previous query, the output displays product IDs like `1` and `2`. Executive decision-makers cannot evaluate product strategy without seeing official product names.

Because `ProductName` resides in the `Products` table while transaction quantities reside in `OrderDetails`, we must *merge* the `Products` table with the `OrderDetails` table before grouping:

```sql
SELECT 
    OD.ProductID, 
    P.ProductName, 
    SUM(OD.Quantity) AS TotalQtySold
FROM OrderDetails AS OD
INNER JOIN Products AS P 
    ON OD.ProductID = P.ProductID
GROUP BY OD.ProductID, P.ProductName
ORDER BY OD.ProductID;
```

*Output (Excerpt):*
| ProductID | ProductName | TotalQtySold |
| :---: | :--- | :---: |
| 1 | Chai | 828 |
| 2 | Chang | 1057 |
| 3 | Aniseed Syrup | 328 |
| 4 | Chef Anton's Cajun Seasoning | 453 |
| 5 | Chef Anton's Gumbo Mix | 298 |

```{warning}
**The Non-Aggregated Column Rule**  
Look closely at the `SELECT` and `GROUP BY` clauses in the query above. Notice that both `OD.ProductID` and `P.ProductName` are explicitly listed in the `GROUP BY` clause!

**CRITICAL RULE**  
*Every un-aggregated column listed in the `SELECT` clause MUST be included in the `GROUP BY` clause.*

Why does this rule exist? If you request `P.ProductName` in `SELECT` but group only by `OD.ProductID`, the SQL engine does not know that a 1-to-1 relationship exists between Product ID and Product Name. If multiple distinct product names were associated with a single ID, SQL would not know which name to display on the single summary output row. Therefore, SQL requires every non-aggregated column in `SELECT` to be listed in `GROUP BY`. Omitting an un-aggregated column from `GROUP BY` results in an immediate syntax error.
```

---

## Descriptive Statistics Across Groups

Aggregate functions are not restricted to simple counts or sums. You can calculate multiple summary statistics for each category.

### Price Variation and Discount Dispersion

In the `OrderDetails` table, Northwind Traders grants percentage discounts on line items. As a result, the actual net price paid per unit (`UnitPrice * (1.0 - Discount)`) varies across historical orders for the same product.

An auditor evaluating discount compliance wants to review the minimum, average, and maximum selling prices for every product line:

```sql
SELECT 
    ProductID,
    MIN(UnitPrice * (1.0 - Discount)) AS MinNetPrice,
    AVG(UnitPrice * (1.0 - Discount)) AS AvgNetPrice,
    MAX(UnitPrice * (1.0 - Discount)) AS MaxNetPrice
FROM OrderDetails
GROUP BY ProductID
ORDER BY ProductID
```

*Output (Excerpt):*
| ProductID | MinNetPrice | AvgNetPrice | MaxNetPrice |
| :---: | :---: | :---: | :---: |
| 1 | \$11.52 | \$15.80 | \$18.00 |
| 2 | \$11.40 | \$16.06 | \$19.00 |
| 3 | \$7.20 | \$9.35 | \$10.00 |
| 4 | \$14.08 | \$19.10 | \$22.00 |
| 5 | \$13.60 | \$18.18 | \$21.35 |

*Managerial Accounting Interpretation:* For Product 1 (`Chai`, list price \$18.00), historical net selling prices ranged from a minimum of \$11.52 to a maximum of \$18.00, averaging \$15.80. Evaluating statistical spread across groups helps decision-makers learn the typical revenue per unit that they can expect. This helps with product portfolio planning, and provides insight into the opportunity cost of scarce warehouse capacity.

---

## Multi-Column Transformations and Period Summaries

Real-world financial reporting frequently requires embedding complex mathematical transformations (`UnitPrice * (1.0 - Discount) * Quantity`) inside aggregate functions and grouping across multiple period columns (`Year`, `Month`).

### 1. Gross Sales Revenue by Product

To compute gross sales revenue by product across the company's entire history, embed the net revenue transformation inside the `SUM()` function:

```sql
SELECT 
    ProductID,
    SUM(UnitPrice * (1.0 - Discount) * Quantity) AS TotalRevenue
FROM OrderDetails
GROUP BY ProductID
```

### 2. Restricting Revenue Summaries to Specific Accounting Periods

Executive managers rarely review all-time historical totals; they require annual or quarterly reporting. 

To evaluate product sales revenue strictly for fiscal year 1997, join the `Orders` table to access `OrderDate`, apply a `WHERE` filter for `YEAR(OrderDate) = 1997`, and group by `ProductID`:

```sql
SELECT 
    OD.ProductID,
    SUM(OD.UnitPrice * (1.0 - OD.Discount) * OD.Quantity) AS Revenue1997
FROM OrderDetails AS OD
INNER JOIN Orders AS O 
    ON OD.OrderID = O.OrderID
WHERE YEAR(O.OrderDate) = 1997
GROUP BY OD.ProductID
```

### 3. Revenue Summaries by Customer

By modifying the grouping key from `ProductID` to `CustomerID`, the exact same underlying transaction data generates a customer sales ranking for 1997:

```sql
SELECT 
    O.CustomerID,
    SUM(OD.UnitPrice * (1.0 - OD.Discount) * OD.Quantity) AS Revenue1997
FROM OrderDetails AS OD
INNER JOIN Orders AS O 
    ON OD.OrderID = O.OrderID
WHERE YEAR(O.OrderDate) = 1997
GROUP BY O.CustomerID
```

### 4. Multi-Period Income Statement Summaries (Year and Month)

Corporate financial statements evaluate performance across multiple time horizons. Suppose the Chief Financial Officer requires a monthly financial performance report listing revenue, cost of goods sold (COGS), and gross margin for every calendar month in the database.

To build this report, we group by two expressions: `YEAR(OrderDate)` and `MONTH(OrderDate)`:

```sql
SELECT 
    YEAR(O.OrderDate) AS [Year],
    MONTH(O.OrderDate) AS [Month],
    SUM(OD.UnitPrice * (1.0 - OD.Discount) * OD.Quantity) AS Revenue,
    SUM(OD.UnitCost * OD.Quantity) AS COGS,
    SUM((OD.UnitPrice * (1.0 - OD.Discount) - OD.UnitCost) * OD.Quantity) 
        AS GrossMargin
FROM OrderDetails AS OD
INNER JOIN Orders AS O 
    ON OD.OrderID = O.OrderID
GROUP BY YEAR(O.OrderDate), MONTH(O.OrderDate)
ORDER BY [Year], [Month]
```

*Output (Excerpt):*
| Year | Month | Revenue | COGS | GrossMargin |
| :---: | :---: | :---: | :---: | :---: |
| 1996 | 7 | \$27,861.90 | \$14,200.00 | \$13,661.90 |
| 1996 | 8 | \$25,485.27 | \$13,100.00 | \$12,385.27 |
| 1996 | 9 | \$26,381.40 | \$13,850.00 | \$12,531.40 |
| 1996 | 10 | \$37,515.73 | \$19,400.00 | \$18,115.73 |
| 1996 | 11 | \$45,600.04 | \$23,200.00 | \$22,400.04 |
| 1996 | 12 | \$45,239.63 | \$23,050.00 | \$22,189.63 |

*How SQL Evaluates Multi-Column Grouping:*
When you group by `YEAR(OrderDate), MONTH(OrderDate)`, SQL creates a distinct summary "pile" for every unique combination of year and month (e.g., `1996-07`, `1996-08`, `1997-01`). It then aggregates all line items falling within that specific month to compute total revenue, total COGS, and total gross margin.

---

## Filtering Aggregated Data: The HAVING Clause

Suppose your sales manager reviews the 1997 product sales summary and asks a follow-up question: *"Filter this report to show ONLY products whose total 1997 revenue exceeded \$20,000."*

Your natural instinct might be to add a condition to the `WHERE` clause:

```sql
-- CAUTION: THIS QUERY PRODUCES A SYNTAX ERROR!
SELECT 
    OD.ProductID,
    SUM(OD.UnitPrice * (1.0 - OD.Discount) * OD.Quantity) AS Revenue
FROM OrderDetails AS OD
INNER JOIN Orders AS O ON OD.OrderID = O.OrderID
-- WHERE is invalid here!
WHERE YEAR(O.OrderDate) = 1997 
  AND SUM(OD.UnitPrice * (1.0 - OD.Discount) * OD.Quantity) > 20000
GROUP BY OD.ProductID;
```

If you attempt to execute this query, SQL Server throws an error: `An aggregate may not appear in the WHERE clause unless it is in a subquery...`

### Why `WHERE` Cannot Filter Aggregate Results

To understand why `WHERE` fails on aggregate totals, you must understand **SQL's internal clause execution order**:

```
+------------------------------------------------------------------------+
|                       SQL EXECUTION SEQUENCE                           |
+------------------------------------------------------------------------+
|  1. FROM / JOIN   --> Identifies source tables and merges rows         |
|  2. WHERE         --> Filters individual detail rows BEFORE grouping   |
|  3. GROUP BY      --> Sorts remaining detail rows into category piles  |
|  4. HAVING        --> Filters aggregated summary piles AFTER grouping  |
|  5. SELECT        --> Constructs output columns and creates aliases    |
|  6. ORDER BY      --> Sorts final summary rows for display             |
+------------------------------------------------------------------------+
```

Look closely at Step 2 (`WHERE`) versus Step 3 (`GROUP BY`):
* The **`WHERE`** clause operates on **individual transaction rows** *before* SQL groups them into piles.
* When Step 2 (`WHERE`) evaluates, SQL has not yet performed Step 3 (`GROUP BY`). The aggregated total `SUM(...)` does not exist yet! 

Therefore, you cannot use `WHERE` to filter on aggregate calculations.

### The Solution: The `HAVING` Clause

To filter aggregated summary results *after* grouping has occurred, use SQL's **`HAVING`** clause. 

The `HAVING` clause functions identically to `WHERE`, but evaluates conditions on aggregated group summary piles during Step 4:

```sql
SELECT 
    OD.ProductID,
    SUM(OD.UnitPrice * (1.0 - OD.Discount) * OD.Quantity) AS Revenue
FROM OrderDetails AS OD
INNER JOIN Orders AS O 
    ON OD.OrderID = O.OrderID
WHERE YEAR(O.OrderDate) = 1997
GROUP BY OD.ProductID
HAVING SUM(OD.UnitPrice * (1.0 - OD.Discount) * OD.Quantity) > 20000
ORDER BY Revenue DESC
```

*Output:*
| ProductID | Revenue |
| :---: | :---: |
| 38 | \$49,198.09 |
| 59 | \$35,775.30 |
| 29 | \$34,755.91 |
| 56 | \$32,604.00 |
| 51 | \$24,570.80 |
| 62 | \$21,638.30 |
| 60 | \$20,505.40 |

### The Alias Execution Rule: `HAVING` vs. `ORDER BY`

Notice that in the `HAVING` clause above, we wrote out the full aggregate expression `HAVING SUM(...) > 20000` rather than using the column alias `HAVING Revenue > 20000`.

Why can't you use the alias `Revenue` in `HAVING`?

Once again, the answer lies in SQL's execution sequence:
* **Step 4 (`HAVING`):** SQL filters summary piles.
* **Step 5 (`SELECT`):** SQL constructs output columns and assigns the alias `Revenue`.
* **Step 6 (`ORDER BY`):** SQL sorts the final summary rows.

Because Step 4 (`HAVING`) executes *before* Step 5 (`SELECT`), the alias `Revenue` does not exist yet when `HAVING` evaluates! Attempting to use a `SELECT` alias inside `HAVING` causes an `Invalid column name` error.

By contrast, Step 6 (`ORDER BY`) executes *after* Step 5 (`SELECT`), so `ORDER BY Revenue DESC` works perfectly.

### Summary Comparison: `WHERE` vs. `HAVING`

| Feature | `WHERE` Clause | `HAVING` Clause |
| :--- | :--- | :--- |
| **Purpose** | Filters individual detail rows | Filters aggregated group summary piles |
| **Execution Point** | Executes *before* `GROUP BY` | Executes *after* `GROUP BY` |
| **Can Use Aggregates?** | NO (`WHERE SUM(...)` is invalid) | YES (`HAVING SUM(...) > x` is allowed) |
| **Can Use Aliases?** | NO (Evaluates before `SELECT`) | NO (Evaluates before `SELECT`) |
| **Primary Use Case** | Eliminating irrelevant data (e.g., `WHERE YEAR(Date) = 1997`) | Isolating key summary groups (e.g., `HAVING Revenue > 20000`) |

---

## Summary and Complete SQL Execution Checklist

Grouping and aggregation completes our study of relational data retrieval in SQL. By combining `SELECT`, `FROM`, `JOIN`, `WHERE`, `GROUP BY`, `HAVING`, and `ORDER BY`, you possess a powerful toolkit with which you can answer complex enterprise business questions.

### Key Chapter Takeaways

1. **Grouping Collapses Rows into Category Piles:** The `GROUP BY` clause combines detailed transaction rows into categories based on unique values in grouping columns.
2. **Aggregate Functions Summarize Groups:** Once rows are grouped, aggregate functions (`SUM`, `COUNT`, `AVG`, `MIN`, `MAX`) compute a single summary result per group.
3. **The Non-Aggregated Column Rule:** Every un-aggregated column listed in the `SELECT` clause must be explicitly included in the `GROUP BY` clause to eliminate display ambiguity.
4. **`WHERE` vs. `HAVING`:** Use `WHERE` to filter data rows before grouping; use `HAVING` to filter aggregated summary rows after grouping.
5. **Execution Order Dictates Syntax:** SQL evaluates queries in a strict internal sequence: `FROM/JOIN` $\rightarrow$ `WHERE` $\rightarrow$ `GROUP BY` $\rightarrow$ `HAVING` $\rightarrow$ `SELECT` $\rightarrow$ `ORDER BY`.

### Complete SQL Statement Execution Checklist

Below is the definitive, unchangeable execution order for all seven major clauses in a SQL `SELECT` statement:

```sql
SELECT TOP (n) ColumnList_and_Aggregations
FROM PrimaryTable AS A
[INNER | LEFT] JOIN SecondaryTable AS B 
    ON A.SharedKey = B.SharedKey
WHERE Detail_Row_Filter_Conditions
GROUP BY Grouping_Columns
HAVING Aggregate_Summary_Filter_Conditions
ORDER BY Sort_Columns [ASC | DESC]
```

| Order | Clause | Purpose | Required or Optional? | Primary Accounting / Analytics Role |
| :---: | :--- | :--- | :---: | :--- |
| **1st** | **`FROM`** | Identifies the primary source database table. | **Required** | Defines the foundational transaction ledger. |
| **2nd** | **`JOIN ... ON`** | Links related secondary tables using shared key columns. | Optional | Merges parent entity names (customers, products, employees). |
| **3rd** | **`WHERE`** | Filters individual transaction rows before grouping. | Optional | Restricts analysis to specific fiscal years, territories, or active statuses. |
| **4th** | **`GROUP BY`** | Collapses detail rows into distinct category piles. | Optional | Groups data by customer, product, department, or period. |
| **5th** | **`HAVING`** | Filters aggregated summary category piles after grouping. | Optional | Isolates top-tier revenue items, loss-making categories, or control breaches. |
| **6th** | **`SELECT`** | Constructs output columns, evaluates aggregations, and assigns aliases (`AS`). | **Required** | Formats calculated metrics and reports. |
| **7th** | **`ORDER BY`** | Sorts final summary result rows for display. | Optional | Ranks top revenue drivers, chronological trends, or risk exposures. |